# Strands Agents와 AgentCore Memory(단기 메모리) - MemoryManager 사용


## 소개

이 튜토리얼에서는 **MemoryManager**와 **MemorySessionManager**를 통해 Strands Agents와 AgentCore **단기 메모리**를 사용하여 **Personal Agent**를 구축하는 방법을 살펴봅니다. Agent는 `get_last_k_turns`를 사용하여 세션의 최근 대화를 기억하며, 사용자가 돌아오면 대화를 자연스럽게 이어 갈 수 있습니다.

**참고: 이 예제는 MemoryManager와 MemorySessionManager를 사용하는 단기 메모리 버전입니다.**


### 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화                                                        |
| Agent 유형          | Personal Agent                                                                   |
| Agentic Framework   | Strands Agents                                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory with MemoryManager, AgentInitializedEvent and MessageAddedEvent hooks   |
| 예제 난이도  | 초급                                                                         |

다음 내용을 학습합니다.
- MemoryManager로 대화 연속성을 위한 단기 메모리 사용
- MemorySessionManager로 최근 K개의 대화 turn 검색
- 실시간 정보를 위한 웹 검색 도구 사용
- 세션 관리를 사용하여 대화 기록으로 Agent 초기화
- MemoryClient에서 MemoryManager 아키텍처로 마이그레이션할 때 활용

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
- Python 3.10 이상
- AgentCore Memory 권한이 있는 AWS 자격 증명
- MemoryManager를 지원하는 Amazon Bedrock AgentCore SDK
- Amazon Bedrock 모델에 대한 액세스

먼저 환경을 설정하겠습니다.

## 1단계: 설정 및 가져오기

In [2]:
!pip install -qr requirements.txt

In [3]:
import logging
from datetime import datetime

# 로깅 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("personal-agent")

In [4]:
# Strands Agent에 필요한 모듈 가져오기
import os
from strands import Agent, tool
from strands.hooks import (
    AgentInitializedEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)

# 메모리 관리 모듈 가져오기
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

# 메시지 역할 상수 정의
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

# 구성
REGION = os.getenv("AWS_REGION", "us-east-1")  # Agent용 AWS 리전
ACTOR_ID = "user_123"  # 고유 식별자라면 무엇이든 사용 가능(AgentID, User ID 등)
SESSION_ID = "personal_session_001"  # 고유 세션 식별자

# IAM 역할 생성을 위한 boto3 가져오기

## 2단계: 웹 검색 도구

먼저 Agent에서 사용할 간단한 웹 검색 도구를 생성합니다. 이 부분은 기존 구현과 동일합니다.

In [ ]:
from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS


@tool
def websearch(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.

    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.

    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except DDGSException as e:
        return f"Search error: {e}"
    except Exception as e:
        return f"Search error: {str(e)}"


logger.info("✅ Web search tool ready")

## 3단계: MemoryManager로 Memory 리소스 생성

단기 메모리에서는 MemoryManager를 사용하여 strategy 없이 Memory 리소스를 생성합니다. 여기에는 `get_last_k_turns`로 검색할 수 있는 원시 대화 turn이 저장됩니다.

**참고: 이 섹션에서는 기존 MemoryClient 대신 MemoryManager 아키텍처를 사용합니다.**

In [ ]:
# Memory Manager 초기화
memory_manager = MemoryManager(region_name=REGION)
memory_name = "PersonalAgentMemoryManager"

logger.info(f"✅ MemoryManager initialized for region: {REGION}")
logger.info(f"Memory manager type: {type(memory_manager)}")

# MemoryManager로 Memory 리소스 생성
logger.info(f"Creating memory '{memory_name}' for short-term conversational storage...")

try:
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=[],  # 단기 메모리에는 strategy를 사용하지 않음
        description="Short-term memory for personal agent",
        event_expiry_days=7,  # 단기 메모리 보존 기간
        memory_execution_role_arn=None,  # 단기 메모리에서는 선택 사항
    )
    memory_id = memory.id
    logger.info("✅ Successfully created/retrieved memory with MemoryManager:")
    logger.info(f"   Memory ID: {memory_id}")
    logger.info(f"   Memory Name: {memory.name}")
    logger.info(f"   Memory Status: {memory.status}")

except Exception as e:
    # 향상된 오류 보고와 함께 Memory 생성 오류 처리
    logger.error(f"❌ Memory creation failed: {e}")
    logger.error(f"Error type: {type(e).__name__}")
    import traceback

    traceback.print_exc()

    # 오류 발생 시 정리 - 일부 생성된 Memory 삭제
    if "memory_id" in locals():
        try:
            logger.info(f"Attempting cleanup of partially created memory: {memory_id}")
            memory_manager.delete_memory(memory_id)
            logger.info(f"✅ Successfully cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"❌ Failed to clean up memory: {cleanup_error}")

    # 원래 예외 다시 발생
    raise

## 4단계: Session Manager 초기화

이 섹션에서는 세션 기반 메모리 작업을 위한 MemorySessionManager를 소개하고 actor와 session을 관리할 MemorySession을 생성합니다.

In [ ]:
# Session Memory Manager 초기화
session_manager = MemorySessionManager(memory_id=memory.id, region_name=REGION)

# 특정 actor/session 조합의 Memory Session 생성
user_session = session_manager.create_memory_session(actor_id=ACTOR_ID, session_id=SESSION_ID)

logger.info(f"✅ Session manager initialized for memory: {memory.id}")
logger.info(f"✅ Memory session created for actor: {ACTOR_ID}, session: {SESSION_ID}")
logger.info(f"Session manager type: {type(session_manager)}")
logger.info(f"Memory session type: {type(user_session)}")

## 5단계: Memory Hook Provider

이 단계에서는 MemorySession을 사용하여 메모리 작업을 자동화하는 사용자 지정 `MemoryHookProvider` 클래스를 정의합니다. Hook은 Agent 실행 수명 주기의 특정 시점에 실행되는 특수 함수입니다. 여기서 만드는 Memory Hook에는 두 가지 주요 기능이 있습니다.
1. **최근 대화 불러오기**: `AgentInitializedEvent` Hook을 사용하여 Agent 초기화 시 최근 대화 기록을 자동으로 불러옵니다.
2. **마지막 메시지 저장**: Session Manager를 사용하여 새 대화 메시지를 저장합니다.

**MemoryClient 버전과 비교한 주요 변경 사항:**
- MemoryClient 대신 MemorySession 사용
- tuple 대신 ConversationalMessage 객체 사용
- create_event() 대신 add_turns() 사용
- type safety를 위해 MessageRole enum 사용

In [8]:
class MemoryHookProvider(HookProvider):
    def __init__(self, memory_session: MemorySession):  # 대신 MemorySession을 받음
        self.memory_session = memory_session

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트가 시작될 때 MemorySession으로 최근 대화 기록을 불러옵니다."""
        try:
            # 사전 구성된 Memory Session 사용(actor_id/session_id 불필요)
            recent_turns = self.memory_session.get_last_k_turns(k=5)

            if recent_turns:
                # 대화 기록을 컨텍스트 형식으로 변환
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        # EventMessage 객체와 dict 형식을 모두 처리
                        if hasattr(message, "role") and hasattr(message, "content"):
                            role = message["role"]
                            content = message["content"]
                        else:
                            role = message.get("role", "unknown")
                            content = message.get("content", {}).get("text", "")
                        context_messages.append(f"{role}: {content}")

                context = "\n".join(context_messages)
                # Agent의 system prompt에 컨텍스트 추가
                event.agent.system_prompt += f"\n\nRecent conversation:\n{context}"
                logger.info(f"✅ Loaded {len(recent_turns)} conversation turns using MemorySession")

        except Exception as e:
            logger.error(f"Memory load error: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        """MemorySession으로 메시지를 메모리에 저장합니다."""
        messages = event.agent.messages
        try:
            if messages and len(messages) > 0 and messages[-1]["content"][0].get("text"):
                message_text = messages[-1]["content"][0]["text"]
                message_role = MessageRole.USER if messages[-1]["role"] == "user" else MessageRole.ASSISTANT

                # Memory Session 인스턴스 사용(actor_id/session_id 전달 불필요)
                result = self.memory_session.add_turns(messages=[ConversationalMessage(message_text, message_role)])

                event_id = result["eventId"]
                logger.info(f"✅ Stored message with Event ID: {event_id}, Role: {message_role.value}")

        except Exception as e:
            logger.error(f"Memory save error: {e}")
            import traceback

            logger.error(f"Full traceback: {traceback.format_exc()}")

    def register_hooks(self, registry: HookRegistry):
        # Memory Hook 등록
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        logger.info("✅ Memory hooks registered with MemorySession")

## 6단계: 웹 검색 기능을 갖춘 Personal Agent 생성

이 Agent는 MemorySessionManager에서 생성한 MemorySession과 연동되는 MemoryHookProvider를 사용합니다.

In [ ]:
def create_personal_agent():
    """MemorySession을 사용해 메모리와 웹 검색 기능이 있는 개인 에이전트를 생성합니다."""
    agent = Agent(
        name="PersonalAssistant",
        model="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # 또는 선호하는 모델
        system_prompt=f"""You are a helpful personal assistant with web search capabilities.
        
        You can help with:
        - General questions and information lookup
        - Web searches for current information
        - Personal task management
        
        When you need current information, use the websearch function.
        Today's date: {datetime.today().strftime("%Y-%m-%d")}
        Be friendly and professional.""",
        hooks=[MemoryHookProvider(user_session)],
        tools=[websearch],
    )
    return agent


# Agent 생성
agent = create_personal_agent()
logger.info("✅ Personal agent created with MemorySession and web search")

#### 축하합니다! MemoryManager와 MemorySession을 사용하는 Agent가 준비되었습니다.
## Agent 테스트

In [ ]:
# 메모리가 적용된 대화 테스트
print("=== First Conversation ===")
print("User: My name is Alex and I'm interested in learning about AI.")
print("Agent: ", end="")
agent("My name is Alex and I'm interested in learning about AI.")

In [ ]:
print("User: Can you search for the latest AI trends in 2025?")
print("Agent: ", end="")
agent("Can you search for the latest AI trends in 2025?")

In [ ]:
print("User: I'm particularly interested in machine learning applications.")
print("Agent: ", end="")
agent("I'm particularly interested in machine learning applications.")

## MemorySessionManager를 사용한 메모리 연속성 테스트

메모리 시스템이 올바르게 작동하는지 확인하기 위해 새 Agent 인스턴스를 생성하고 MemorySessionManager를 통해 이전에 저장한 정보에 액세스할 수 있는지 살펴보겠습니다.

In [ ]:
# 새 Agent 인스턴스 생성(사용자가 돌아오는 상황 모의)
print("=== User Returns - New Session ===")
new_agent = create_personal_agent()

# 메모리 연속성 테스트
print("User: What was my name again?")
print("Agent: ", end="")
new_agent("What was my name again?")

print("User: Can you search for more information about machine learning?")
print("Agent: ", end="")
new_agent("Can you search for more information about machine learning?")

## MemorySession으로 저장된 메모리 확인

In [ ]:
# MemorySession을 사용하여 저장된 내용 확인
print("=== Memory Contents ===")
recent_turns = user_session.get_last_k_turns(k=3)

for i, turn in enumerate(recent_turns, 1):
    print(f"Turn {i}:")
    for message in turn:
        role = message["role"]
        content = (
            message["content"]["text"][:100] + "..."
            if len(message["content"]["text"]) > 100
            else message["content"]["text"]
        )
        print(f"  {role}: {content}")
    print()

## 요약

이 튜토리얼에서는 MemorySessionManager와 MemorySession을 모두 사용하여 Personal Agent를 구축했습니다. 학습한 내용은 다음과 같습니다.

- **MemorySessionManager**: 여러 세션의 메모리 작업을 관리하는 상위 수준 Manager
- **MemorySession**: 반복적인 parameter 전달을 없애는 세션 전용 interface입니다. MemorySession을 사용하면 모든 method에 actor_id/session_id를 전달할 필요가 없습니다.
- **Type Safety**: 세션 생성 시 특정 actor/session에 바인딩됩니다.
- **향상된 캡슐화**: 세션별 작업이 세션 객체 안에 포함됩니다.
- **Memory Hook**: Agent Hook이 세션 기반 아키텍처와 연동됩니다.
- **대화 연속성**: MemoryManager와 MemorySession으로 단기 메모리 기능을 유지합니다.

### MemorySession의 주요 이점
1. **간소화된 API**: 모든 method 호출에 actor_id/session_id를 전달할 필요가 없습니다.
2. **사전 구성된 컨텍스트**: 세션 생성 시 특정 actor/session에 바인딩됩니다.
3. **일관된 interface**: 모든 세션 작업에서 동일하게 사전 구성된 컨텍스트를 사용합니다.


## 리소스 정리(선택 사항)

In [ ]:
# MemoryManager로 Memory 리소스를 삭제하려면 주석 해제
# try:
#     memory_manager.delete_memory(memory_id)
#     logger.info(f"✅ Deleted memory: {memory_id}")
# except Exception as e:
#     logger.error(f"Failed to delete memory: {e}")